# FSI AIxData Challenge 2024 | Final modeling pipeline

이 노트북은 당시 사용한 **CTGAN 증강 → Stratified 5-Fold 검증 설계 → 언더샘플링 비율 9·10 → LightGBM·XGBoost 블렌딩 → 제출 ZIP 생성** 흐름을 포트폴리오용으로 정리한 실행본입니다.

> 원본 대회 데이터와 과거 제출 파일은 공개 저장소에 포함하지 않습니다. `data/README.md`의 입력 계약을 만족한 로컬 환경에서 실행하세요.


## 1. 실행 전제

- 데이터는 `data/train.csv`, `data/test.csv`, `data/sample_submission.csv`에 둡니다.
- CTGAN은 클래스별로 1,000 epoch를 학습하므로 실행 비용이 큽니다. `RUN_FOLD_VALIDATION`은 기본적으로 꺼져 있습니다.
- 아래 함수는 검증 fold의 학습 데이터만 증강하도록 작성했습니다. 이 노트북은 데이터가 없으므로 출력값을 저장하지 않은 상태입니다.


In [ ]:
from __future__ import annotations

from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
import random

import numpy as np
import pandas as pd
import torch
from lightgbm import LGBMClassifier
from sdv.metadata import SingleTableMetadata
from sdv.single_table import CTGANSynthesizer
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from tqdm.auto import tqdm
from xgboost import XGBClassifier


In [ ]:
RANDOM_STATE = 736665
N_SPLITS = 5
EXPECTED_TRAIN_ROWS = 120_000
EXPECTED_TEST_ROWS = 120_000
EXPECTED_MAJOR_CLASS = 'm'
EXPECTED_MAJOR_CLASS_ROWS = 118_800
CTGAN_EPOCHS = 1_000
CTGAN_FIT_ROWS_PER_CLASS = 100
CTGAN_ROWS_PER_CLASS = 1_000
SAVE_CTGAN_LOSS = False  # True로 바꾸면 실제 학습 loss를 outputs/ctgan_loss/에 HTML로 저장
RUN_FOLD_VALIDATION = False  # True면 CTGAN을 fold마다 다시 학습하므로 실행 시간이 크게 늘어납니다.
UNDER_SAMPLING_RATIOS = (9, 10)
EXCLUDED_SYNTHETIC_LABELS = {'m'}  # 당시 최종 분류 학습에서 제외한 합성 라벨

DROP_IDENTIFIER_COLUMNS = [
    'Customer_personal_identifier',
    'Customer_identification_number',
    'Account_account_number',
    'Recipient_Account_Number',
    'Another_Person_Account',
]
CATEGORICAL_COLUMNS = [
    'Customer_Gender', 'Customer_credit_rating', 'Customer_loan_type',
    'Account_account_type', 'Channel', 'Operating_System', 'Error_Code',
    'Type_General_Automatic', 'Access_Medium',
]
DROP_HIGH_CARDINALITY_COLUMNS = [
    'Customer_registration_datetime', 'Account_creation_datetime',
    'Transaction_Datetime', 'IP_Address', 'MAC_Address', 'Location',
    'Last_atm_transaction_datetime', 'Last_bank_branch_transaction_datetime',
    'Transaction_resumed_date',
]

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)


## 2. 데이터 위치와 입력 검증


In [ ]:
def locate_data_dir() -> Path:
    expected = ('train.csv', 'test.csv', 'sample_submission.csv')
    candidates = (Path.cwd() / 'data', Path.cwd().parent / 'data')
    for candidate in candidates:
        if all((candidate / name).is_file() for name in expected):
            return candidate
    searched = ', '.join(str(path.resolve()) for path in candidates)
    raise FileNotFoundError(f'대회 입력 파일을 찾지 못했습니다. 확인한 경로: {searched}')


def load_competition_data(data_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    train = pd.read_csv(data_dir / 'train.csv')
    test = pd.read_csv(data_dir / 'test.csv')
    sample_submission = pd.read_csv(data_dir / 'sample_submission.csv')
    if 'Fraud_Type' not in train.columns:
        raise ValueError('train.csv에 Fraud_Type 컬럼이 필요합니다.')
    if 'Fraud_Type' not in sample_submission.columns:
        raise ValueError('sample_submission.csv에 Fraud_Type 컬럼이 필요합니다.')
    return train, test, sample_submission


def report_input_profile(train: pd.DataFrame, test: pd.DataFrame) -> pd.DataFrame:
    target_counts = train['Fraud_Type'].value_counts()
    profile = pd.DataFrame(
        {
            'observed': [len(train), len(test), target_counts.index[0], target_counts.iloc[0]],
            'reference': [
                EXPECTED_TRAIN_ROWS, EXPECTED_TEST_ROWS,
                EXPECTED_MAJOR_CLASS, EXPECTED_MAJOR_CLASS_ROWS,
            ],
        },
        index=['train rows', 'test rows', 'major class', 'major class rows'],
    )
    if profile['observed'].tolist() != profile['reference'].tolist():
        print('참고: 입력 프로필이 당시 대회 데이터와 다릅니다. 실험은 실행할 수 있지만 결과 비교에는 사용하지 마세요.')
    return profile


DATA_DIR = locate_data_dir()
OUTPUT_DIR = DATA_DIR.parent / 'outputs'
train_raw, test_raw, sample_submission = load_competition_data(DATA_DIR)
report_input_profile(train_raw, test_raw)


## 3. CTGAN 증강

아래 함수는 전달받은 학습 프레임 내부에서만 합성 데이터를 만듭니다. 5-Fold 평가를 수행할 때는 반드시 각 train fold를 인자로 넣어야 합니다.


In [ ]:
CTGAN_SDTYPES = {
    'Account_initial_balance': 'numerical',
    'Account_balance': 'numerical',
    'Customer_identification_number': 'categorical',
    'Customer_personal_identifier': 'categorical',
    'Account_account_number': 'categorical',
    'IP_Address': 'ipv4_address',
    'Location': 'categorical',
    'Recipient_Account_Number': 'categorical',
    'Fraud_Type': 'categorical',
    'Time_difference_seconds': 'numerical',
    'Customer_Birthyear': 'numerical',
}


def generate_synthetic_data(train_frame: pd.DataFrame) -> pd.DataFrame:
    generated_parts: list[pd.DataFrame] = []
    for fraud_type in tqdm(sorted(train_frame['Fraud_Type'].unique()), desc='CTGAN by class'):
        subset = train_frame.loc[train_frame['Fraud_Type'].eq(fraud_type)].copy()
        subset = subset.sample(
            n=min(CTGAN_FIT_ROWS_PER_CLASS, len(subset)),
            random_state=RANDOM_STATE,
        )
        subset['Time_difference_seconds'] = pd.to_timedelta(
            subset['Time_difference'], errors='coerce'
        ).dt.total_seconds()
        subset = subset.drop(columns='Time_difference')

        metadata = SingleTableMetadata()
        metadata.detect_from_dataframe(subset)
        metadata.set_primary_key(None)
        for column, sdtype in CTGAN_SDTYPES.items():
            if column in subset.columns:
                metadata.update_column(column_name=column, sdtype=sdtype)

        synthesizer = CTGANSynthesizer(
            metadata,
            enforce_min_max_values=False,
            locales=['ko_KR'],
            epochs=CTGAN_EPOCHS,
        )
        synthesizer.fit(subset)
        if SAVE_CTGAN_LOSS:
            loss_dir = OUTPUT_DIR / 'ctgan_loss'
            loss_dir.mkdir(parents=True, exist_ok=True)
            synthesizer.get_loss_values_plot().write_html(loss_dir / f'{fraud_type}_loss.html')
        synthetic_subset = synthesizer.sample(num_rows=CTGAN_ROWS_PER_CLASS)
        synthetic_subset['Time_difference'] = pd.to_timedelta(
            synthetic_subset.pop('Time_difference_seconds'), unit='s'
        )
        generated_parts.append(synthetic_subset)

    return pd.concat(generated_parts, ignore_index=True)


## 4. 전처리와 언더샘플링

원-핫 인코더는 반드시 학습 데이터에서 fit하고, 검증·테스트 데이터에는 transform만 적용합니다. 다수 클래스는 인코딩된 숫자가 아니라 관측 빈도가 가장 큰 원본 라벨을 기준으로 선택합니다.


In [ ]:
def make_one_hot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False)
    except TypeError:  # scikit-learn < 1.2
        return OneHotEncoder(handle_unknown='ignore', drop='first', sparse=False)


def clean_features(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.drop(columns=['ID', *DROP_IDENTIFIER_COLUMNS], errors='ignore').copy()
    if 'Time_difference' in result.columns:
        result['Time_difference_seconds'] = pd.to_timedelta(
            result.pop('Time_difference'), errors='coerce'
        ).dt.total_seconds()
    return result.drop(columns=DROP_HIGH_CARDINALITY_COLUMNS, errors='ignore')


def prepare_feature_matrices(
    fit_frame: pd.DataFrame, *transform_frames: pd.DataFrame
) -> tuple[pd.DataFrame, ...]:
    frames = [clean_features(frame) for frame in (fit_frame, *transform_frames)]
    active_categorical = [column for column in CATEGORICAL_COLUMNS if column in frames[0].columns]

    for column in active_categorical:
        encoder = make_one_hot_encoder()
        encoded_frames = [
            encoder.fit_transform(frames[0][[column]])
        ] + [encoder.transform(frame[[column]]) for frame in frames[1:]]
        feature_names = encoder.get_feature_names_out([column])
        frames = [
            pd.concat(
                [frame.drop(columns=column).reset_index(drop=True),
                 pd.DataFrame(encoded, columns=feature_names)],
                axis=1,
            )
            for frame, encoded in zip(frames, encoded_frames)
        ]

    remaining_object_columns = frames[0].select_dtypes(include=['object', 'string']).columns
    frames = [frame.drop(columns=remaining_object_columns, errors='ignore') for frame in frames]
    reference_columns = frames[0].columns
    return tuple(frame.reindex(columns=reference_columns, fill_value=0) for frame in frames)


def undersample_majority(labeled_frame: pd.DataFrame, ratio: int) -> pd.DataFrame:
    label_counts = labeled_frame['Fraud_Type'].value_counts()
    majority_label = label_counts.index[0]
    majority = labeled_frame.loc[labeled_frame['Fraud_Type'].eq(majority_label)]
    minority = labeled_frame.loc[~labeled_frame['Fraud_Type'].eq(majority_label)]
    per_class_target = int(len(minority) / (len(label_counts) - 1) * ratio)
    sampled_majority = majority.sample(
        n=min(len(majority), per_class_target), random_state=RANDOM_STATE
    )
    return pd.concat([sampled_majority, minority], ignore_index=True)


## 5. 5-Fold 검증 설계

각 fold에서 `fold_train`만 CTGAN에 전달하고, 증강·전처리·언더샘플링·학습 후 untouched `fold_valid`의 Macro F1을 측정합니다. 실행 비용이 크므로 필요한 경우에만 호출합니다.


In [ ]:
def fit_blended_models(
    labeled_train: pd.DataFrame, prediction_frame: pd.DataFrame
) -> tuple[np.ndarray, LabelEncoder]:
    label_encoder = LabelEncoder()
    labels = label_encoder.fit_transform(labeled_train['Fraud_Type'])
    x_train, x_prediction = prepare_feature_matrices(
        labeled_train.drop(columns='Fraud_Type'), prediction_frame
    )
    prepared = x_train.copy()
    prepared['Fraud_Type'] = labels

    probabilities = []
    for ratio in UNDER_SAMPLING_RATIOS:
        sampled = undersample_majority(prepared, ratio)
        x_sampled = sampled.drop(columns='Fraud_Type')
        y_sampled = sampled['Fraud_Type']
        models = (
            LGBMClassifier(random_state=RANDOM_STATE, verbose=-1),
            XGBClassifier(random_state=RANDOM_STATE),
        )
        for model in models:
            model.fit(x_sampled, y_sampled)
            probabilities.append(model.predict_proba(x_prediction))

    return np.mean(probabilities, axis=0), label_encoder


def evaluate_with_stratified_folds(train_frame: pd.DataFrame) -> pd.DataFrame:
    splitter = StratifiedKFold(
        n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE
    )
    scores = []
    for fold, (train_index, valid_index) in enumerate(
        splitter.split(train_frame, train_frame['Fraud_Type']), start=1
    ):
        fold_train = train_frame.iloc[train_index].drop(columns='ID', errors='ignore').copy()
        fold_valid = train_frame.iloc[valid_index].drop(columns='ID', errors='ignore').copy()
        fold_synthetic = generate_synthetic_data(fold_train)
        classifier_synthetic = fold_synthetic.loc[
            ~fold_synthetic['Fraud_Type'].isin(EXCLUDED_SYNTHETIC_LABELS)
        ]
        augmented_train = pd.concat([fold_train, classifier_synthetic], ignore_index=True)
        probabilities, encoder = fit_blended_models(
            augmented_train, fold_valid.drop(columns='Fraud_Type')
        )
        predictions = encoder.inverse_transform(probabilities.argmax(axis=1))
        score = f1_score(fold_valid['Fraud_Type'], predictions, average='macro')
        scores.append({'fold': fold, 'macro_f1': score})
    return pd.DataFrame(scores)

if RUN_FOLD_VALIDATION:
    fold_scores = evaluate_with_stratified_folds(train_raw)
    display(fold_scores)


## 6. 전체 학습·예측·제출 ZIP 생성

최종 단계에서는 전체 학습 데이터로 합성 데이터를 생성한 뒤, 당시 최종 분류 학습에서 제외했던 `m` 라벨의 합성 행을 제외하고 4개 모델을 학습합니다. 합성 데이터 제출 파일에는 전체 생성 행을 사용합니다. ZIP 생성 전후로 제출물 행·컬럼·클래스별 생성 행 수·압축 내부 파일 목록을 검증합니다.


In [ ]:
train_for_modeling = train_raw.drop(columns='ID', errors='ignore').copy()
synthetic_data = generate_synthetic_data(train_for_modeling)
classifier_synthetic = synthetic_data.loc[
    ~synthetic_data['Fraud_Type'].isin(EXCLUDED_SYNTHETIC_LABELS)
]
augmented_train = pd.concat([train_for_modeling, classifier_synthetic], ignore_index=True)

probabilities, label_encoder = fit_blended_models(augmented_train, test_raw)
predicted_labels = label_encoder.inverse_transform(probabilities.argmax(axis=1))

clf_submission = sample_submission.copy()
clf_submission['Fraud_Type'] = predicted_labels

def validate_submission_frames(
    classifier_frame: pd.DataFrame,
    synthetic_frame: pd.DataFrame,
    submission_template: pd.DataFrame,
    training_columns: pd.Index,
    training_labels: pd.Index,
) -> pd.DataFrame:
    if list(classifier_frame.columns) != list(submission_template.columns):
        raise ValueError('clf_submission.csv의 컬럼 순서가 sample_submission.csv와 다릅니다.')
    if len(classifier_frame) != len(submission_template):
        raise ValueError('clf_submission.csv의 행 수가 sample_submission.csv와 다릅니다.')

    expected_columns = list(training_columns)
    if set(synthetic_frame.columns) != set(expected_columns):
        raise ValueError('syn_submission.csv의 컬럼 구성이 학습 데이터와 다릅니다.')
    synthetic_frame = synthetic_frame.reindex(columns=expected_columns)
    observed_counts = synthetic_frame['Fraud_Type'].value_counts().reindex(training_labels, fill_value=0)
    if not observed_counts.eq(CTGAN_ROWS_PER_CLASS).all():
        raise ValueError('syn_submission.csv는 클래스별 1,000행이어야 합니다.')
    if len(synthetic_frame) != len(training_labels) * CTGAN_ROWS_PER_CLASS:
        raise ValueError('syn_submission.csv의 전체 행 수가 올바르지 않습니다.')
    return synthetic_frame

synthetic_data = validate_submission_frames(
    clf_submission, synthetic_data, sample_submission, train_for_modeling.columns,
    pd.Index(train_raw['Fraud_Type'].unique()),
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
clf_path = OUTPUT_DIR / 'clf_submission.csv'
syn_path = OUTPUT_DIR / 'syn_submission.csv'
zip_path = OUTPUT_DIR / 'fsi_aixdata_submission.zip'

clf_submission.to_csv(clf_path, index=False, encoding='utf-8-sig')
synthetic_data.to_csv(syn_path, index=False, encoding='utf-8-sig')
with ZipFile(zip_path, mode='w', compression=ZIP_DEFLATED) as archive:
    archive.write(clf_path, arcname='clf_submission.csv')
    archive.write(syn_path, arcname='syn_submission.csv')

with ZipFile(zip_path) as archive:
    if archive.namelist() != ['clf_submission.csv', 'syn_submission.csv']:
        raise ValueError('ZIP에는 clf_submission.csv와 syn_submission.csv만 포함되어야 합니다.')

print(f'Created: {zip_path.resolve()}')


## Takeaways

- 검증 점수를 신뢰하려면 합성 데이터 생성도 fold 내부에서 끝나야 합니다.
- 다수 클래스 비율을 고정하지 않고 후보 비율을 비교한 뒤, 서로 다른 비율의 모델을 결합했습니다.
- 예측 라벨, 합성 데이터, ZIP 구조를 분리해 제출 단계의 실수를 줄였습니다.
